# Ensemble de Predicciones 202108

Este notebook ensambla las probabilidades de 4 experimentos diferentes para 202108:

1. `comp2_entrega1_202108_zlgbm`
2. `comp2_entrega2_202108_under0.05_zlgbm`
3. `comp2_entrega3_202108_under0.1_zlgbm`
4. `comp2_entrega4_202108_under0.05_zero_zlgbm`

**Estrategia:** Promedio simple de las probabilidades de cada modelo.


In [31]:
import polars as pl
from pathlib import Path
import numpy as np

print("="*80)
print("ENSEMBLE DE PREDICCIONES 202108")
print("="*80)


ENSEMBLE DE PREDICCIONES 202108


In [32]:
# Step 1: Define experiments to ensemble
print("\n1. Defining experiments...")
print("-"*80)

experiments = [
    "comp2_entrega1_202108_zlgbm",
    "comp2_entrega2_202108_under0.05_zlgbm",
    "comp2_entrega3_202108_under0.1_zlgbm",
    "comp2_entrega4_202108_under0.05_zero_zlgbm"
]

output_dir = Path("../output")

print(f"\n  Experiments to ensemble ({len(experiments)}):")
for i, exp in enumerate(experiments, 1):
    print(f"    {i}. {exp}")



1. Defining experiments...
--------------------------------------------------------------------------------

  Experiments to ensemble (4):
    1. comp2_entrega1_202108_zlgbm
    2. comp2_entrega2_202108_under0.05_zlgbm
    3. comp2_entrega3_202108_under0.1_zlgbm
    4. comp2_entrega4_202108_under0.05_zero_zlgbm


In [33]:
# Step 2: Load all predictions
print("\n2. Loading predictions from each experiment...")
print("-"*80)

predictions = {}
missing = []

for exp in experiments:
    pred_file = output_dir / exp / "prediccion.txt"
    
    if not pred_file.exists():
        print(f"  [WARNING] Missing: {exp}")
        missing.append(exp)
        continue
    
    df = pl.read_csv(pred_file, separator="\t")
    predictions[exp] = df
    
    print(f"  [OK] Loaded {exp}")
    print(f"       Records: {len(df):,}")
    print(f"       Prob range: [{df['prob'].min():.6f}, {df['prob'].max():.6f}]")
    print(f"       Mean prob: {df['prob'].mean():.6f}")

if missing:
    print(f"\n  [ERROR] Missing {len(missing)} experiments:")
    for exp in missing:
        print(f"    - {exp}")
    raise FileNotFoundError(f"Cannot proceed without all experiments")

print(f"\n  [SUCCESS] All {len(predictions)} experiments loaded!")



2. Loading predictions from each experiment...
--------------------------------------------------------------------------------
  [OK] Loaded comp2_entrega1_202108_zlgbm
       Records: 164,822
       Prob range: [0.000000, 1.000000]
       Mean prob: 0.023839
  [OK] Loaded comp2_entrega2_202108_under0.05_zlgbm
       Records: 164,822
       Prob range: [0.000000, 1.000000]
       Mean prob: 0.037481
  [OK] Loaded comp2_entrega3_202108_under0.1_zlgbm
       Records: 164,822
       Prob range: [0.000000, 0.999999]
       Mean prob: 0.022889
  [OK] Loaded comp2_entrega4_202108_under0.05_zero_zlgbm
       Records: 164,822
       Prob range: [0.000000, 1.000000]
       Mean prob: 0.037047

  [SUCCESS] All 4 experiments loaded!


In [34]:
# Step 3: Verify all predictions have same clients
print("\n3. Verifying client alignment...")
print("-"*80)

# Get reference client list from first experiment
ref_exp = experiments[0]
ref_clients = predictions[ref_exp]["numero_de_cliente"].sort()

all_aligned = True
for exp in experiments[1:]:
    exp_clients = predictions[exp]["numero_de_cliente"].sort()
    
    if not ref_clients.equals(exp_clients):
        print(f"  [ERROR] Client mismatch in {exp}")
        all_aligned = False
    else:
        print(f"  [OK] {exp} - clients aligned")

if not all_aligned:
    raise ValueError("Not all experiments have the same clients!")

print(f"\n  [SUCCESS] All experiments have {len(ref_clients):,} clients in same order")



3. Verifying client alignment...
--------------------------------------------------------------------------------
  [OK] comp2_entrega2_202108_under0.05_zlgbm - clients aligned
  [OK] comp2_entrega3_202108_under0.1_zlgbm - clients aligned
  [OK] comp2_entrega4_202108_under0.05_zero_zlgbm - clients aligned

  [SUCCESS] All experiments have 164,822 clients in same order


In [35]:
# Step 4: Ensemble predictions (simple average)
print("\n4. Creating ensemble predictions...")
print("-"*80)

# Start with first experiment as base
df_ensemble = predictions[experiments[0]].select(["numero_de_cliente"]).sort("numero_de_cliente")

# Collect all probabilities
prob_columns = []
for exp in experiments:
    df_exp = predictions[exp].sort("numero_de_cliente")
    prob_col_name = f"prob_{exp}"
    df_ensemble = df_ensemble.with_columns(
        df_exp["prob"].alias(prob_col_name)
    )
    prob_columns.append(prob_col_name)

# Calculate average probability
df_ensemble = df_ensemble.with_columns([
    pl.mean_horizontal(prob_columns).alias("prob_ensemble")
])

print(f"\n  Individual model probabilities:")
for exp, prob_col in zip(experiments, prob_columns):
    mean_prob = df_ensemble[prob_col].mean()
    print(f"    {exp}: {mean_prob:.6f}")

print(f"\n  Ensemble probability:")
print(f"    Mean: {df_ensemble['prob_ensemble'].mean():.6f}")
print(f"    Min:  {df_ensemble['prob_ensemble'].min():.6f}")
print(f"    Max:  {df_ensemble['prob_ensemble'].max():.6f}")
print(f"    Std:  {df_ensemble['prob_ensemble'].std():.6f}")



4. Creating ensemble predictions...
--------------------------------------------------------------------------------

  Individual model probabilities:
    comp2_entrega1_202108_zlgbm: 0.023839
    comp2_entrega2_202108_under0.05_zlgbm: 0.037481
    comp2_entrega3_202108_under0.1_zlgbm: 0.022889
    comp2_entrega4_202108_under0.05_zero_zlgbm: 0.037047

  Ensemble probability:
    Mean: 0.030314
    Min:  0.000000
    Max:  1.000000
    Std:  0.151278


In [36]:
# Step 5: Generate final submission file
print("\n5. Generating final submission file...")
print("-"*80)

# Create final dataframe with only numero_de_cliente and prob
df_final = df_ensemble.select([
    "numero_de_cliente",
    pl.col("prob_ensemble").alias("prob")
]).sort("numero_de_cliente")

# Save to output directory
output_file = Path("../output/ensemble_202108_4models.txt")
df_final.write_csv(output_file, separator="\t")

print(f"\n  [SUCCESS] Ensemble predictions saved!")
print(f"  File: {output_file}")
print(f"  Records: {len(df_final):,}")
print(f"\n  Preview (top 10 by probability):")
print(df_final.sort("prob", descending=True).head(10))



5. Generating final submission file...
--------------------------------------------------------------------------------

  [SUCCESS] Ensemble predictions saved!
  File: ..\output\ensemble_202108_4models.txt
  Records: 164,822

  Preview (top 10 by probability):
shape: (10, 2)
┌───────────────────┬──────────┐
│ numero_de_cliente ┆ prob     │
│ ---               ┆ ---      │
│ i64               ┆ f64      │
╞═══════════════════╪══════════╡
│ 1248124471        ┆ 1.0      │
│ 709048487         ┆ 1.0      │
│ 1327932710        ┆ 1.0      │
│ 738415924         ┆ 1.0      │
│ 1323711255        ┆ 0.999999 │
│ 595282122         ┆ 0.999999 │
│ 839768428         ┆ 0.999999 │
│ 700219427         ┆ 0.999999 │
│ 1032325045        ┆ 0.999999 │
│ 1488309466        ┆ 0.999999 │
└───────────────────┴──────────┘


In [37]:
# Step 6: Compare individual models vs ensemble
print("\n6. Comparing individual models vs ensemble...")
print("-"*80)

# For each model, show correlation with ensemble
print(f"\n  Correlation with ensemble:")
for exp, prob_col in zip(experiments, prob_columns):
    corr = df_ensemble.select([
        pl.corr(prob_col, "prob_ensemble").alias("correlation")
    ])["correlation"][0]
    print(f"    {exp}: {corr:.4f}")

# Show top 11000 clients for each model
print(f"\n  Top 11,000 clients overlap:")
top_11k_ensemble = set(df_final.sort("prob", descending=True).head(11000)["numero_de_cliente"])

for exp in experiments:
    df_exp = predictions[exp].sort("prob", descending=True)
    top_11k_exp = set(df_exp.head(11000)["numero_de_cliente"])
    overlap = len(top_11k_ensemble & top_11k_exp)
    overlap_pct = overlap / 11000 * 100
    print(f"    {exp}: {overlap:,} ({overlap_pct:.1f}%)")



6. Comparing individual models vs ensemble...
--------------------------------------------------------------------------------

  Correlation with ensemble:
    comp2_entrega1_202108_zlgbm: 0.9470
    comp2_entrega2_202108_under0.05_zlgbm: 0.9705
    comp2_entrega3_202108_under0.1_zlgbm: 0.9528
    comp2_entrega4_202108_under0.05_zero_zlgbm: 0.9718

  Top 11,000 clients overlap:
    comp2_entrega1_202108_zlgbm: 9,714 (88.3%)
    comp2_entrega2_202108_under0.05_zlgbm: 10,486 (95.3%)
    comp2_entrega3_202108_under0.1_zlgbm: 9,958 (90.5%)
    comp2_entrega4_202108_under0.05_zero_zlgbm: 10,464 (95.1%)


In [38]:
print("\n" + "="*80)
print("ENSEMBLE COMPLETE!")
print("="*80)
print(f"\nIntermediate file: output/ensemble_202108_4models.txt")
print(f"Next: Generate Kaggle submission file...")



ENSEMBLE COMPLETE!

Intermediate file: output/ensemble_202108_4models.txt
Next: Generate Kaggle submission file...


In [39]:
# Step 7: Generate Kaggle submission file (COMMENTED - already generated)
# print("\n7. Generating Kaggle submission file...")
# print("-"*80)

# # Sort by probability descending and take top 11,000
# df_kaggle = df_final.sort("prob", descending=True).head(11000)

# # Select only numero_de_cliente column (Kaggle format)
# df_kaggle = df_kaggle.select(["numero_de_cliente"])

# # Save to Kaggle directory
# kaggle_dir = Path("../output/kaggle/recortado")
# kaggle_dir.mkdir(exist_ok=True)

# kaggle_file = kaggle_dir / "ensemble_4models_202108.csv"
# df_kaggle.write_csv(kaggle_file, include_header=False)

# print(f"\n  [SUCCESS] Kaggle submission file created!")
# print(f"  File: {kaggle_file}")
# print(f"  Records: {len(df_kaggle):,} (top 11,000 by probability)")
# print(f"\n  Preview (first 20 clients):")
# print(df_kaggle.head(20))

# print(f"\n  Probability range of selected clients:")
# selected_probs = df_final.sort("prob", descending=True).head(11000)["prob"]
# print(f"    Min:  {selected_probs.min():.6f}")
# print(f"    Max:  {selected_probs.max():.6f}")
# print(f"    Mean: {selected_probs.mean():.6f}")

# print("\n" + "="*80)
# print("READY FOR KAGGLE SUBMISSION!")
# print("="*80)
# print(f"\nFile to upload: {kaggle_file}")
# print(f"Format: CSV with 11,000 client IDs (no header)")

print("\n[INFO] Kaggle submission file (11k) already generated and moved to output/kaggle/recortado/")



[INFO] Kaggle submission file (11k) already generated and moved to output/kaggle/recortado/


In [40]:
# Step 8: Generate Kaggle submission file with 11,500 envíos
print("\n8. Generating Kaggle submission file (11,500 envíos)...")
print("-"*80)

# Sort by probability descending and take top 11,500
df_kaggle_11500 = df_final.sort("prob", descending=True).head(11500)

# Select only numero_de_cliente column (Kaggle format)
df_kaggle_11500 = df_kaggle_11500.select(["numero_de_cliente"])

# Save to Kaggle directory
kaggle_dir = Path("../output/kaggle/recortado")
kaggle_dir.mkdir(exist_ok=True)

kaggle_file_11500 = kaggle_dir / "ensemble_4models_202108_11500.csv"
df_kaggle_11500.write_csv(kaggle_file_11500, include_header=False)

print(f"\n  [SUCCESS] Kaggle submission file created!")
print(f"  File: {kaggle_file_11500}")
print(f"  Records: {len(df_kaggle_11500):,} (top 11,500 by probability)")
print(f"\n  Preview (first 20 clients):")
print(df_kaggle_11500.head(20))

print(f"\n  Probability range of selected clients:")
selected_probs_11500 = df_final.sort("prob", descending=True).head(11500)["prob"]
print(f"    Min:  {selected_probs_11500.min():.6f}")
print(f"    Max:  {selected_probs_11500.max():.6f}")
print(f"    Mean: {selected_probs_11500.mean():.6f}")

# Compare with 11k cutoff
print(f"\n  Comparison with 11,000 cutoff:")
selected_probs_11000 = df_final.sort("prob", descending=True).head(11000)["prob"]
prob_at_11000 = selected_probs_11000[-1]
prob_at_11500 = selected_probs_11500[-1]
print(f"    Probability at 11,000th client: {prob_at_11000:.6f}")
print(f"    Probability at 11,500th client: {prob_at_11500:.6f}")
print(f"    Difference: {prob_at_11000 - prob_at_11500:.6f}")

print("\n" + "="*80)
print("READY FOR KAGGLE SUBMISSION (11,500 ENVÍOS)!")
print("="*80)
print(f"\nFile to upload: {kaggle_file_11500}")
print(f"Format: CSV with 11,500 client IDs (no header)")



8. Generating Kaggle submission file (11,500 envíos)...
--------------------------------------------------------------------------------

  [SUCCESS] Kaggle submission file created!
  File: ..\output\kaggle\recortado\ensemble_4models_202108_11500.csv
  Records: 11,500 (top 11,500 by probability)

  Preview (first 20 clients):
shape: (20, 1)
┌───────────────────┐
│ numero_de_cliente │
│ ---               │
│ i64               │
╞═══════════════════╡
│ 1248124471        │
│ 709048487         │
│ 1327932710        │
│ 738415924         │
│ 1323711255        │
│ …                 │
│ 873972753         │
│ 572310972         │
│ 614189119         │
│ 601652670         │
│ 254111380         │
└───────────────────┘

  Probability range of selected clients:
    Min:  0.005962
    Max:  1.000000
    Mean: 0.433397

  Comparison with 11,000 cutoff:
    Probability at 11,000th client: 0.008058
    Probability at 11,500th client: 0.005962
    Difference: 0.002096

READY FOR KAGGLE SUBMISSION (11,5

In [41]:
# Step 9: Generate ensemble WITHOUT entrega1 (only 3 models)
print("\n9. Generating ensemble WITHOUT entrega1 (3 models only)...")
print("-"*80)

# Use only models 2, 3, and 4
experiments_no_entrega1 = [
    "comp2_entrega2_202108_under0.05_zlgbm",
    "comp2_entrega3_202108_under0.1_zlgbm",
    "comp2_entrega4_202108_under0.05_zero_zlgbm"
]

print(f"\n  Models in this ensemble:")
for i, exp in enumerate(experiments_no_entrega1, 1):
    print(f"    {i}. {exp}")

# Create ensemble dataframe
df_ensemble_3models = predictions[experiments_no_entrega1[0]].select(["numero_de_cliente"]).sort("numero_de_cliente")

# Collect probabilities from 3 models
prob_columns_3models = []
for exp in experiments_no_entrega1:
    df_exp = predictions[exp].sort("numero_de_cliente")
    prob_col_name = f"prob_{exp}"
    df_ensemble_3models = df_ensemble_3models.with_columns(
        df_exp["prob"].alias(prob_col_name)
    )
    prob_columns_3models.append(prob_col_name)

# Calculate average probability (3 models)
df_ensemble_3models = df_ensemble_3models.with_columns([
    pl.mean_horizontal(prob_columns_3models).alias("prob_ensemble_3models")
])

print(f"\n  Individual model probabilities:")
for exp, prob_col in zip(experiments_no_entrega1, prob_columns_3models):
    mean_prob = df_ensemble_3models[prob_col].mean()
    print(f"    {exp}: {mean_prob:.6f}")

print(f"\n  Ensemble probability (3 models):")
print(f"    Mean: {df_ensemble_3models['prob_ensemble_3models'].mean():.6f}")
print(f"    Min:  {df_ensemble_3models['prob_ensemble_3models'].min():.6f}")
print(f"    Max:  {df_ensemble_3models['prob_ensemble_3models'].max():.6f}")
print(f"    Std:  {df_ensemble_3models['prob_ensemble_3models'].std():.6f}")

# Create final dataframe
df_final_3models = df_ensemble_3models.select([
    "numero_de_cliente",
    pl.col("prob_ensemble_3models").alias("prob")
]).sort("numero_de_cliente")

# Generate Kaggle submission (11,000 envíos)
df_kaggle_3models = df_final_3models.sort("prob", descending=True).head(11000)
df_kaggle_3models = df_kaggle_3models.select(["numero_de_cliente"])

# Save to Kaggle directory
kaggle_file_3models = kaggle_dir / "ensemble_3models_no_entrega1_202108.csv"
df_kaggle_3models.write_csv(kaggle_file_3models, include_header=False)

print(f"\n  [SUCCESS] Kaggle submission file created!")
print(f"  File: {kaggle_file_3models}")
print(f"  Records: {len(df_kaggle_3models):,} (top 11,000 by probability)")

print(f"\n  Probability range of selected clients:")
selected_probs_3models = df_final_3models.sort("prob", descending=True).head(11000)["prob"]
print(f"    Min:  {selected_probs_3models.min():.6f}")
print(f"    Max:  {selected_probs_3models.max():.6f}")
print(f"    Mean: {selected_probs_3models.mean():.6f}")

# Compare with 4-model ensemble
print(f"\n  Comparison with 4-model ensemble:")
top_11k_4models = set(df_final.sort("prob", descending=True).head(11000)["numero_de_cliente"])
top_11k_3models = set(df_kaggle_3models["numero_de_cliente"])
overlap = len(top_11k_4models & top_11k_3models)
overlap_pct = overlap / 11000 * 100
print(f"    Overlap: {overlap:,} clients ({overlap_pct:.1f}%)")
print(f"    Different: {11000 - overlap:,} clients ({100 - overlap_pct:.1f}%)")

print("\n" + "="*80)
print("READY FOR KAGGLE SUBMISSION (3 MODELS, NO ENTREGA1)!")
print("="*80)
print(f"\nFile to upload: {kaggle_file_3models}")
print(f"Format: CSV with 11,000 client IDs (no header)")



9. Generating ensemble WITHOUT entrega1 (3 models only)...
--------------------------------------------------------------------------------

  Models in this ensemble:
    1. comp2_entrega2_202108_under0.05_zlgbm
    2. comp2_entrega3_202108_under0.1_zlgbm
    3. comp2_entrega4_202108_under0.05_zero_zlgbm

  Individual model probabilities:
    comp2_entrega2_202108_under0.05_zlgbm: 0.037481
    comp2_entrega3_202108_under0.1_zlgbm: 0.022889
    comp2_entrega4_202108_under0.05_zero_zlgbm: 0.037047

  Ensemble probability (3 models):
    Mean: 0.032472
    Min:  0.000000
    Max:  1.000000
    Std:  0.157659

  [SUCCESS] Kaggle submission file created!
  File: ..\output\kaggle\recortado\ensemble_3models_no_entrega1_202108.csv
  Records: 11,000 (top 11,000 by probability)

  Probability range of selected clients:
    Min:  0.009860
    Max:  1.000000
    Mean: 0.484808

  Comparison with 4-model ensemble:
    Overlap: 10,911 clients (99.2%)
    Different: 89 clients (0.8%)

READY FOR KAG